## Setup and Imports

In [1]:
# Install required packages if not available
!pip install torch torchvision scikit-learn pandas matplotlib seaborn thop tqdm -q

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, resnet50

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import time
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## Configuration and Hyperparameters

In [2]:
# Configuration
CONFIG = {
    'batch_sizes': [16, 32],
    'optimizers': ['SGD', 'Adam'],
    'learning_rates': [0.001, 0.0001],
    'epochs_options': [5, 10],  # Two different epoch values
    'pin_memory_options': [False, True],
    'USE_AMP': True,  # Constant
    'train_ratio': 0.7,
    'val_ratio': 0.1,
    'test_ratio': 0.2,
}

# For storing results
results_q1a = []
results_q1b = []
results_q2 = []

In [3]:
def get_transforms():
    """Get transforms for MNIST/FashionMNIST to work with ResNet"""
    transform = transforms.Compose([
        transforms.Resize((224, 224)),  # ResNet expects 224x224
        transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return transform

def load_dataset(dataset_name='MNIST'):
    """Load MNIST or FashionMNIST dataset with 70-10-20 split"""
    transform = get_transforms()

    if dataset_name == 'MNIST':
        full_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:  # FashionMNIST
        full_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # Combine train and test for custom split
    full_dataset = torch.utils.data.ConcatDataset([full_train, test_dataset])
    total_size = len(full_dataset)

    train_size = int(CONFIG['train_ratio'] * total_size)
    val_size = int(CONFIG['val_ratio'] * total_size)
    test_size = total_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"Dataset: {dataset_name}")
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    return train_dataset, val_dataset, test_dataset

def create_dataloaders(train_dataset, val_dataset, test_dataset, batch_size, pin_memory=True):
    """Create data loaders"""
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=2, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                             num_workers=2, pin_memory=pin_memory)
    return train_loader, val_loader, test_loader

## Model Definition

In [4]:
def get_model(model_name='ResNet18', num_classes=10, pretrained=False):
    """Get ResNet model without pretrained weights"""
    if model_name == 'ResNet18':
        model = resnet18(weights=None)  # pretrained=False
    elif model_name == 'ResNet50':
        model = resnet50(weights=None)  # pretrained=False
    else:
        raise ValueError(f"Unknown model: {model_name}")

    # Modify final layer for 10 classes
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

def get_optimizer(model, opt_name, lr):
    """Get optimizer"""
    if opt_name == 'SGD':
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        return optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unknown optimizer: {opt_name}")

## Training and Evaluation Functions

In [5]:
def train_epoch(model, train_loader, criterion, optimizer, device, use_amp=True):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    scaler = GradScaler() if use_amp else None

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        if use_amp:
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(train_loader), 100. * correct / total

def evaluate(model, data_loader, criterion, device):
    """Evaluate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(data_loader), 100. * correct / total

In [6]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device,
                epochs, use_amp=True, verbose=True):
    """Full training loop with history"""
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    start_time = time.time()

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, use_amp)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if verbose:
            print(f"Epoch {epoch+1}/{epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
                  f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    total_time = (time.time() - start_time) * 1000  # in ms
    return history, total_time

def plot_training_history(history, title):
    """Plot training and validation curves"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss plot
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].legend()
    axes[0].grid(True)

    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
    axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

## FLOPs Calculation

In [7]:
try:
    from thop import profile
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False
    print("thop not available, FLOPs calculation will use manual estimation")

def calculate_flops(model, input_size=(1, 3, 224, 224), device='cpu'):
    """Calculate FLOPs for a model"""
    model = model.to(device)
    model.eval()

    if THOP_AVAILABLE:
        dummy_input = torch.randn(input_size).to(device)
        flops, params = profile(model, inputs=(dummy_input,), verbose=False)
        return flops, params
    else:
        # Manual estimation based on model type
        total_params = sum(p.numel() for p in model.parameters())
        # Rough estimation: FLOPs ≈ 2 * MACs ≈ 2 * params * input_elements
        flops = total_params * 2
        return flops, total_params

def format_flops(flops):
    """Format FLOPs to readable string"""
    if flops >= 1e9:
        return f"{flops/1e9:.2f} GFLOPs"
    elif flops >= 1e6:
        return f"{flops/1e6:.2f} MFLOPs"
    else:
        return f"{flops:.2f} FLOPs"

thop not available, FLOPs calculation will use manual estimation


---
# Q1(a): Training ResNet-18 and ResNet-50 on MNIST and FashionMNIST

Training deep learning models with:
- 70%-10%-20% train-val-test split
- Varying batch sizes, optimizers, learning rates
- USE_AMP = True (constant)
- pretrained = False

In [8]:
# Load datasets
print("Loading MNIST dataset...")
mnist_train, mnist_val, mnist_test = load_dataset('MNIST')

print("\nLoading FashionMNIST dataset...")
fashion_train, fashion_val, fashion_test = load_dataset('FashionMNIST')

Loading MNIST dataset...


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 485kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.41MB/s]


Dataset: MNIST
Train: 49000, Val: 7000, Test: 14000

Loading FashionMNIST dataset...


100%|██████████| 26.4M/26.4M [00:02<00:00, 12.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 211kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.94MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.1MB/s]

Dataset: FashionMNIST
Train: 49000, Val: 7000, Test: 14000


---
# Q2: CPU vs GPU Performance Comparison on FashionMNIST

Comparing training time and FLOPs for ResNet-18 and ResNet-50 on CPU and GPU

In [9]:
def run_experiment_q2(compute_device, batch_size, optimizer_name, lr, epochs=3):
    """Run Q2 experiment comparing CPU vs GPU"""
    results = []

    # Use subset for faster experiments
    train_subset = torch.utils.data.Subset(fashion_train, range(min(5000, len(fashion_train))))
    val_subset = torch.utils.data.Subset(fashion_val, range(min(1000, len(fashion_val))))
    test_subset = torch.utils.data.Subset(fashion_test, range(min(2000, len(fashion_test))))

    train_loader, val_loader, test_loader = create_dataloaders(
        train_subset, val_subset, test_subset, batch_size, pin_memory=True
    )

    for model_name in ['ResNet18', 'ResNet50']:
        print(f"\n{'='*60}")
        print(f"Q2: {compute_device.upper()} | {model_name} | Batch: {batch_size} | {optimizer_name} | LR: {lr}")
        print('='*60)

        # Create model
        model = get_model(model_name, num_classes=10, pretrained=False)
        target_device = torch.device(compute_device)
        model = model.to(target_device)

        # Calculate FLOPs
        flops, params = calculate_flops(model, device=compute_device)
        print(f"FLOPs: {format_flops(flops)}, Parameters: {params:,}")

        # Setup training
        criterion = nn.CrossEntropyLoss()
        optimizer = get_optimizer(model, optimizer_name, lr)

        # Train (use AMP only for GPU)
        use_amp = (compute_device == 'cuda') and CONFIG['USE_AMP']
        history, train_time = train_model(
            model, train_loader, val_loader, criterion, optimizer,
            target_device, epochs, use_amp=use_amp, verbose=True
        )

        # Test evaluation
        test_loss, test_acc = evaluate(model, test_loader, criterion, target_device)
        print(f"Test Accuracy: {test_acc:.2f}%")

        results.append({
            'Compute': compute_device.upper(),
            'Model': model_name,
            'Batch Size': batch_size,
            'Optimizer': optimizer_name,
            'Learning Rate': lr,
            'Test Accuracy (%)': round(test_acc, 2),
            'Train Time (ms)': round(train_time, 2),
            'FLOPs': flops,
            'FLOPs (formatted)': format_flops(flops)
        })

        # Clear GPU memory
        if compute_device == 'cuda':
            torch.cuda.empty_cache()

    return results

In [10]:
# Q2 Experiments - CPU vs GPU comparison
q2_results = []

# Configuration as per assignment
q2_configs = [
    {'batch_size': 16, 'optimizer': 'SGD', 'lr': 0.001},
    {'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.001},
]

# Run on CPU
# print("\n" + "#"*80)
# print("RUNNING CPU EXPERIMENTS")
# print("#"*80)
# for config in q2_configs:
#     results = run_experiment_q2('cpu', config['batch_size'], config['optimizer'], config['lr'], epochs=3)
#     q2_results.extend(results)

# Run on GPU (if available)
if torch.cuda.is_available():
    print("\n" + "#"*80)
    print("RUNNING GPU EXPERIMENTS")
    print("#"*80)
    for config in q2_configs:
        results = run_experiment_q2('cuda', config['batch_size'], config['optimizer'], config['lr'], epochs=3)
        q2_results.extend(results)
else:
    print("\nGPU not available. Skipping GPU experiments.")


################################################################################
RUNNING GPU EXPERIMENTS
################################################################################

Q2: CUDA | ResNet18 | Batch: 16 | SGD | LR: 0.001
FLOPs: 22.36 MFLOPs, Parameters: 11,181,642
Epoch 1/3: Train Loss: 1.3739, Train Acc: 51.16%, Val Loss: 0.9175, Val Acc: 67.60%
Epoch 2/3: Train Loss: 0.7597, Train Acc: 72.76%, Val Loss: 0.7016, Val Acc: 73.50%
Epoch 3/3: Train Loss: 0.6025, Train Acc: 78.28%, Val Loss: 1.0183, Val Acc: 57.30%
Test Accuracy: 56.20%

Q2: CUDA | ResNet50 | Batch: 16 | SGD | LR: 0.001
FLOPs: 47.06 MFLOPs, Parameters: 23,528,522
Epoch 1/3: Train Loss: 1.7588, Train Acc: 34.96%, Val Loss: 1.5007, Val Acc: 45.80%
Epoch 2/3: Train Loss: 1.0217, Train Acc: 63.26%, Val Loss: 0.9187, Val Acc: 64.30%
Epoch 3/3: Train Loss: 0.8524, Train Acc: 69.50%, Val Loss: 0.8189, Val Acc: 67.30%
Test Accuracy: 67.40%

Q2: CUDA | ResNet18 | Batch: 16 | Adam | LR: 0.001
FLOPs: 22.36 MFLOPs, Pa

In [12]:
# Run on CPU
print("\n" + "#"*80)
print("RUNNING CPU EXPERIMENTS")
print("#"*80)
for config in q2_configs:
    results = run_experiment_q2('cpu', config['batch_size'], config['optimizer'], config['lr'], epochs=3)
    q2_results.extend(results)



################################################################################
RUNNING CPU EXPERIMENTS
################################################################################

Q2: CPU | ResNet18 | Batch: 16 | SGD | LR: 0.001
FLOPs: 22.36 MFLOPs, Parameters: 11,181,642
Epoch 1/3: Train Loss: 1.3411, Train Acc: 52.68%, Val Loss: 0.8469, Val Acc: 70.40%
Epoch 2/3: Train Loss: 0.7250, Train Acc: 74.30%, Val Loss: 0.6281, Val Acc: 77.40%
Epoch 3/3: Train Loss: 0.5833, Train Acc: 78.68%, Val Loss: 0.5319, Val Acc: 81.20%
Test Accuracy: 79.45%

Q2: CPU | ResNet50 | Batch: 16 | SGD | LR: 0.001
FLOPs: 47.06 MFLOPs, Parameters: 23,528,522


KeyboardInterrupt: 

In [ ]:
# Q2 Results Summary
df_q2 = pd.DataFrame(q2_results)

print("\n" + "="*100)
print("Q2 RESULTS - CPU vs GPU PERFORMANCE COMPARISON (FashionMNIST)")
print("="*100)
print(df_q2[['Compute', 'Model', 'Batch Size', 'Optimizer', 'Learning Rate',
             'Test Accuracy (%)', 'Train Time (ms)', 'FLOPs (formatted)']].to_string(index=False))

In [ ]:
# Q2 Visualization
if len(df_q2) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Training Time Comparison
    df_q2['Config'] = df_q2['Compute'] + '_' + df_q2['Optimizer']
    pivot_time = df_q2.pivot(index='Model', columns='Config', values='Train Time (ms)')
    pivot_time.plot(kind='bar', ax=axes[0])
    axes[0].set_title('Training Time Comparison')
    axes[0].set_ylabel('Time (ms)')
    axes[0].tick_params(axis='x', rotation=0)
    axes[0].legend(title='Compute_Optimizer')

    # Accuracy Comparison
    pivot_acc = df_q2.pivot(index='Model', columns='Config', values='Test Accuracy (%)')
    pivot_acc.plot(kind='bar', ax=axes[1])
    axes[1].set_title('Test Accuracy Comparison')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].tick_params(axis='x', rotation=0)
    axes[1].legend(title='Compute_Optimizer')

    # FLOPs Comparison
    flops_data = df_q2.groupby('Model')['FLOPs'].first().reset_index()
    axes[2].bar(flops_data['Model'], flops_data['FLOPs'] / 1e9, color=['steelblue', 'coral'])
    axes[2].set_title('FLOPs Comparison')
    axes[2].set_ylabel('GFLOPs')
    axes[2].set_xlabel('Model')

    plt.tight_layout()
    plt.savefig('q2_cpu_gpu_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()